# BARC Sentinel-2 field-year baseline

This notebook joins the provided BARC labels to public Sentinel-2 L2A observations, computes transparent field-level NDVI features, and evaluates a small Random Forest with **farm-grouped folds**. It is an evaluation baseline for Maryland BARC fields, not evidence of transfer to California or a national production model. The small, imbalanced labels support a pipeline demonstration only; probability scores are not calibrated.

It uses labeled field-years only. Blank labels are omitted. The GeoPackage contains 44 labeled field-years on 34 polygons; the 5 farms are kept only as CV groups and are not model features. Quick Stats is an aggregate reference and is not used as a field label.

In [ ]:
%pip install geopandas matplotlib numpy odc-stac planetary-computer pystac-client rasterio scikit-learn pandas xarray

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import planetary_computer
from odc.stac import load
from pystac_client import Client
from rasterio.features import geometry_mask
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (balanced_accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd()
GPKG = ROOT / '04_Evaluation' / 'BARC_Corn.2021-2023.gpkg'
LAYER = 'barc_silage_grain_corn20212023'
OUTPUT = ROOT / 'data' / 'processed' / 'sentinel2'
PROJECTED_CRS = 'EPSG:26918'  # NAD83 / UTM zone 18N, meters
MAX_SCENE_CLOUD = 85
VALID_SCL = [4, 5]  # retain vegetation and bare soil; reject water/cloud/shadow/unclassified
OUTPUT.mkdir(parents=True, exist_ok=True)
if not GPKG.exists():
    raise FileNotFoundError(f'Expected {GPKG}')

## Load the reference labels

Only `Grain` and `Silage` values are labels. We retain `FARM_ID` for grouping validation, but exclude it from the feature matrix.

In [ ]:
fields = gpd.read_file(GPKG, layer=LAYER)
LABEL_COLUMNS = {2021: 'Corn_2021', 2022: 'Corn_2022', 2023: 'Corn_2023'}
labels = pd.concat([
    fields[['OBJECTID', 'FARM_ID', 'ACRES', column]]
      .rename(columns={column: 'harvest_type'})
      .assign(year=year)
      .dropna(subset=['harvest_type'])
      .query("harvest_type in ['Grain', 'Silage']")
      .rename(columns={'OBJECTID': 'field_id', 'FARM_ID': 'farm_group', 'ACRES': 'acres'})
      [['field_id', 'farm_group', 'year', 'harvest_type', 'acres']]
    for year, column in LABEL_COLUMNS.items()
], ignore_index=True).sort_values(['year', 'field_id']).reset_index(drop=True)
print(f'Fields in source: {len(fields)}')
print(f'Labeled field-years: {len(labels)}; unique fields: {labels.field_id.nunique()}; farms: {labels.farm_group.nunique()}')
display(labels.groupby(['year', 'harvest_type']).size().unstack(fill_value=0))

fields = fields.to_crs(PROJECTED_CRS)
labeled_ids = labels.field_id.unique()
field_geometries = fields[fields.OBJECTID.isin(labeled_ids)].set_index('OBJECTID').geometry
aoi = field_geometries.unary_union
aoi_wgs84 = gpd.GeoSeries([aoi], crs=PROJECTED_CRS).to_crs('EPSG:4326').iloc[0]

## Find public scenes

Query one July–November window per label year. The scene-wide cloud filter is only a retrieval aid; the pixel-level SCL mask below is the data-quality filter. A missing or cloudy image remains a missing observation, not a zero NDVI value.

In [ ]:
catalog = Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')
scene_items = {}
scene_rows = []
for year in sorted(labels.year.unique()):
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        intersects=aoi_wgs84.__geo_interface__,
        datetime=f'{year}-07-01/{year}-12-01',
        query={'eo:cloud_cover': {'lt': MAX_SCENE_CLOUD}},
    )
    items = list(search.items())
    scene_items[year] = items
    for item in items:
        scene_rows.append({
            'year': year, 'item_id': item.id,
            'datetime': item.datetime.isoformat() if item.datetime else '',
            'scene_cloud_percent': item.properties.get('eo:cloud_cover'),
            'collection': item.collection_id,
        })
    print(f'{year}: {len(items)} candidate scenes')

scene_manifest = pd.DataFrame(scene_rows)
scene_manifest.to_csv(OUTPUT / 'barc_sentinel2_scene_manifest.csv', index=False)
if scene_manifest.empty:
    raise RuntimeError('No scenes found. Check catalog access, AOI, dates, or cloud filter.')
display(scene_manifest.groupby('year').agg(scenes=('item_id', 'count'),
                                            median_scene_cloud=('scene_cloud_percent', 'median')))

## Build field-level NDVI series

We use the field polygon itself, not one centroid pixel. Each row records the clear-pixel share so low-coverage observations can be inspected or excluded later. The output is a derived time series; no imagery is copied into the repository.

In [ ]:
series_rows = []
for year, items in scene_items.items():
    if not items:
        continue
    for item in items:
        planetary_computer.sign_inplace(item)
    cube = load(
        items, bands=['B04', 'B08', 'SCL'],
        geopolygon=aoi_wgs84.__geo_interface__,
        crs=PROJECTED_CRS, resolution=10, groupby='solar_day',
        chunks={'time': 1, 'x': 512, 'y': 512},
    )
    denominator = cube.B08 + cube.B04
    ndvi = ((cube.B08 - cube.B04) / denominator).where(denominator > 0)
    scl_valid = cube.SCL.isin(VALID_SCL)
    geobox = cube.odc.geobox
    for field_id in labeled_ids:
        geom = field_geometries.loc[field_id]
        inside = geometry_mask(
            [geom.__geo_interface__], out_shape=geobox.shape,
            transform=geobox.transform, invert=True, all_touched=False,
        )
        pixel_count = int(inside.sum())
        if pixel_count == 0:
            warnings.warn(f'Field {field_id} has no 10 m pixel centers in the raster grid.')
            continue
        inside_da = xr.DataArray(inside, dims=('y', 'x'), coords={'y': cube.y, 'x': cube.x})
        valid = scl_valid & inside_da
        field_curve = ndvi.where(valid).median(dim=('y', 'x'), skipna=True).compute()
        valid_fraction = (valid.sum(dim=('y', 'x')) / pixel_count).compute()
        dates = pd.to_datetime(field_curve.time.values)
        for date, value, fraction in zip(dates, field_curve.values, valid_fraction.values):
            if np.isfinite(value):
                series_rows.append({
                    'field_id': field_id, 'year': year, 'date': date.date().isoformat(),
                    'ndvi_median': float(value), 'valid_pixel_fraction': float(fraction),
                    'inside_pixel_count': pixel_count,
                })
    del cube

timeseries = pd.DataFrame(series_rows).sort_values(['field_id', 'year', 'date'])
timeseries.to_csv(OUTPUT / 'barc_sentinel2_ndvi_timeseries.csv', index=False)
print(f'Valid field-date observations: {len(timeseries):,}')
display(timeseries.groupby(['year']).agg(fields=('field_id', 'nunique'),
                                          field_dates=('date', 'size'),
                                          median_clear_share=('valid_pixel_fraction', 'median')))

In [ ]:
# Inspect a few labeled curves before modeling; color denotes reference label.
labeled_series = timeseries.merge(labels, on=['field_id', 'year'], how='inner')
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, year in zip(axes, sorted(labels.year.unique())):
    subset = labeled_series[labeled_series.year == year]
    for (field_id, harvest_type), group in subset.groupby(['field_id', 'harvest_type']):
        ax.plot(pd.to_datetime(group.date), group.ndvi_median,
                color={'Grain': '#c69c00', 'Silage': '#2f7d32'}[harvest_type],
                alpha=0.65, linewidth=1)
    ax.set_title(str(year)); ax.set_xlabel('Date'); ax.grid(alpha=0.2)
axes[0].set_ylabel('Field median NDVI')
fig.suptitle('BARC reference fields: Grain gold, Silage green')
plt.tight_layout(); plt.show()

## Features and farm-held-out evaluation

This intentionally starts with a few interpretable seasonal summaries. Median imputation is inside each training fold. A five-fold split holds out whole farms; farm ID is never a model feature. This tests transfer to another farm within the BARC site, not transfer to another county or state.

In [ ]:
def summarize_curve(group):
    group = group.copy()
    group['date'] = pd.to_datetime(group.date)
    group = group.sort_values('date')
    ndvi = group.ndvi_median.to_numpy(dtype=float)
    dates = group.date.to_numpy()
    peak_pos = int(np.nanargmax(ndvi))
    peak_date = pd.Timestamp(dates[peak_pos])
    early = group[group.date < f'{int(group.year.iloc[0])}-09-01'].ndvi_median
    late = group[group.date >= f'{int(group.year.iloc[0])}-09-01'].ndvi_median
    after_peak = group[group.date >= peak_date]
    slope = np.nan
    if len(after_peak) >= 2:
        x = (after_peak.date - peak_date).dt.days.to_numpy(dtype=float)
        slope = float(np.polyfit(x, after_peak.ndvi_median.to_numpy(dtype=float), 1)[0])
    return pd.Series({
        'valid_obs_count': len(group),
        'median_valid_pixel_fraction': group.valid_pixel_fraction.median(),
        'max_observation_gap_days': group.date.diff().dt.days.max(),
        'peak_ndvi': float(ndvi[peak_pos]),
        'peak_day_of_year': peak_date.dayofyear,
        'early_ndvi_median_jul_aug': early.median(),
        'late_ndvi_median_sep_nov': late.median(),
        'late_minus_early_ndvi': late.median() - early.median(),
        'post_peak_slope_per_day': slope,
        'observations_after_peak': int((group.date > peak_date).sum()),
    })

features = (timeseries.groupby(['field_id', 'year'], sort=True)
            .apply(summarize_curve).reset_index())
model_data = features.merge(labels, on=['field_id', 'year'], how='inner')
model_data.to_csv(OUTPUT / 'barc_sentinel2_field_features.csv', index=False)
print(f'Feature rows matched to labels: {len(model_data)} / {len(labels)}')
display(model_data.groupby('harvest_type').valid_obs_count.describe())
FEATURE_COLUMNS = [
    'valid_obs_count', 'median_valid_pixel_fraction', 'max_observation_gap_days',
    'peak_ndvi', 'peak_day_of_year', 'early_ndvi_median_jul_aug',
    'late_ndvi_median_sep_nov', 'late_minus_early_ndvi',
    'post_peak_slope_per_day', 'observations_after_peak',
]
model_data = model_data.dropna(subset=['harvest_type', 'farm_group']).reset_index(drop=True)
X = model_data[FEATURE_COLUMNS]
y = model_data.harvest_type
groups = model_data.farm_group
print('Evaluation rows:', len(model_data), '| unique farms:', groups.nunique())

In [ ]:
splitter = GroupKFold(n_splits=groups.nunique())
oof_model = np.empty(len(y), dtype=object)
oof_dummy = np.empty(len(y), dtype=object)
oof_silage_probability = np.full(len(y), np.nan, dtype=float)
fold_rows = []
for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
    if y.iloc[train_idx].nunique() < 2:
        raise ValueError(f'Fold {fold} training set has only one class; grouped evaluation is not valid.')
    model = make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                               class_weight='balanced', random_state=17),
    )
    dummy = make_pipeline(SimpleImputer(strategy='median'), DummyClassifier(strategy='most_frequent'))
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    dummy.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_model[test_idx] = model.predict(X.iloc[test_idx])
    silage_column = list(model[-1].classes_).index('Silage')
    oof_silage_probability[test_idx] = model.predict_proba(X.iloc[test_idx])[:, silage_column]
    oof_dummy[test_idx] = dummy.predict(X.iloc[test_idx])
    fold_rows.append({
        'fold': fold, 'held_out_farms': ','.join(map(str, sorted(groups.iloc[test_idx].unique()))),
        'test_fields': model_data.iloc[test_idx].field_id.nunique(),
        'test_field_years': len(test_idx),
        'test_grain': int((y.iloc[test_idx] == 'Grain').sum()),
        'test_silage': int((y.iloc[test_idx] == 'Silage').sum()),
    })

def report_predictions(name, predictions):
    print(f'\n{name}')
    print('Balanced accuracy:', round(balanced_accuracy_score(y, predictions), 3))
    print('Confusion matrix (rows=true [Grain, Silage], columns=predicted [Grain, Silage]):')
    display(pd.DataFrame(confusion_matrix(y, predictions, labels=['Grain', 'Silage']),
                         index=['true Grain', 'true Silage'], columns=['pred Grain', 'pred Silage']))
    print(classification_report(y, predictions, labels=['Grain', 'Silage'], zero_division=0, digits=3))

display(pd.DataFrame(fold_rows))
report_predictions('Farm-held-out Random Forest', oof_model)
report_predictions('Farm-held-out majority baseline', oof_dummy)

# Out-of-fold probabilities are uncalibrated; use them only to inspect ranking
# and area-weighted error on these held-out BARC field-years.
oof = model_data[['field_id', 'year', 'farm_group', 'harvest_type', 'acres']].copy()
oof['silage_probability_oof'] = oof_silage_probability
oof['predicted_type_oof'] = oof_model
oof['true_silage_acres'] = np.where(oof.harvest_type.eq('Silage'), oof.acres, 0.0)
oof['estimated_silage_acres'] = oof.acres * oof.silage_probability_oof
oof.to_csv(OUTPUT / 'barc_sentinel2_oof_predictions.csv', index=False)
print('OOF silage acres absolute error:', round(abs(oof.true_silage_acres.sum() - oof.estimated_silage_acres.sum()), 2))
display(oof.sort_values('silage_probability_oof').head(10))

## Interpretation and limits

- Keep the exact field-year count, class balance, per-farm fold counts, and confusion matrix with any reported score.
- The class counts are small and imbalanced. Do not present this single-site validation as a national accuracy estimate.
- Scene-wide cloud cover is not field cloud cover. The per-field valid-pixel share and maximum observation gap must accompany the curve.
- These are timing/greenness features. They do not prove the harvest method, and they do not estimate silage tons or dairy demand.
- Keep the scene manifest and feature CSV with the submission. For reproducibility, record the notebook run date and any changed catalog, date window, cloud threshold, band list, or SCL classes.